# ОКС: модели, Optuna и контролируемые эксперименты

Переносимая версия расчётного кода. Выводы очищены. Результаты записываются только в `runs/acs`. Исходные снимки в `results/` не изменяются. Полный поиск в 02 требует существенного времени. См. README и docs/REPRODUCIBILITY.md.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "clinical-mortality-mpl"))
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "repository_support.py").exists())
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from repository_support import initialize
OUT = initialize("acs")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name): (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))
def figsave(name):
    plt.tight_layout(); plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight"); plt.savefig(OUT / (name + ".svg"), bbox_inches="tight"); plt.show(); plt.close()
print("Working directory:", OUT, "Python:", platform.python_version(), "seed:", BASE_SEED, VERSIONS)


In [ ]:
import time, warnings, joblib, optuna, sqlite3, math
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix, roc_curve, precision_score, recall_score, f1_score, accuracy_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits
threadpool_limits(2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
meta=json.loads((OUT/"01_конфигурация.json").read_text())
assert sha(OUT/"01_данные.csv") == meta["data_sha256"]
# Public data integrity is checked through metadata/data SHA-256.
D=pd.read_csv(OUT/"01_данные.csv")
FEATURES=meta["features"]
X=D[FEATURES];y=D.hospital_death.to_numpy();groups=D.group_id.to_numpy()
FAMILIES=["LR","RF","CatBoost","XGBoost"]
STRATEGIES=["all_real","class_weights","stratified_4to1"]
BASE={"LR":{"C":1.0},"RF":{"n_estimators":500,"max_depth":5,"min_samples_leaf":1,"max_features":"sqrt"},"CatBoost":{"iterations":300,"depth":4,"learning_rate":0.05,"l2_leaf_reg":3.0},"XGBoost":{"n_estimators":300,"max_depth":3,"learning_rate":0.05,"min_child_weight":1.0,"subsample":1.0,"colsample_bytree":1.0,"reg_lambda":1.0,"reg_alpha":0.0}}
CONFIG={"version":"oks_crp_indicator_restored_v3","seed":BASE_SEED,"outer_folds":3,"outer_repeats":3,"inner_folds":3,"main_trials":50,"conditional_trials":20,"families":FAMILIES,"imbalance":STRATEGIES,"missing_indicator":[False,True],"feature_rule":"full_15_features_fixed","selection":"max mean inner AP; ties within 1e-6: min Brier, then LR RF CatBoost XGBoost","threshold":"max inner OOF balanced accuracy; ties greater sensitivity","baseline":BASE,"data_sha256":meta["data_sha256"],"versions":VERSIONS,"spaces":{"LR":"C log 1e-3..100","RF":"n_estimators 200..800 step100; depth2..8; leaf1,2,4,8,16; features sqrt,0.5,1.0","CatBoost":"iterations100..800 step100; depth2..6; lr log.01...2; l2 log.01..100","XGBoost":"estimators100..800 step100; depth1..5; lr log.01...2; child1..20; sample/col .6..1; lambda log.001..100; alpha0..5"}}
code='\n'.join(c.source for c in __import__('nbformat').read(OUT/"02_модели_Optuna_и_эксперименты.ipynb",as_version=4).cells if c.cell_type=="code")
CONFIG["code_sha256"]=hashlib.sha256(code.encode()).hexdigest()
CONFIG["excluded_missing_indicators"]=[]
CONFIG["restored_missing_indicator"]="crp"
CONFIG["parent_run_id"]="oks_no_crp_indicator_v2_d138be90f9f1"
RUN_ID="oks_crp_indicator_restored_v3_"+hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
CONFIG["run_id"]=RUN_ID
savejson(CONFIG,"02_протокол.json")
STORAGE="sqlite:///"+str(OUT/"02_optuna_missing_crp_возвращён.sqlite")
def log(message):
    line=f"{datetime.datetime.now(datetime.timezone.utc).isoformat()} {RUN_ID} {message}"
    print(line,flush=True)
    with (OUT/"02_журнал_запуска.txt").open("a") as f:f.write(line+"\n")
log("Protocol fixed before outer computations")
SAMPLING_AUDIT=[];INNER_SPLITS=[]
def splits(a,b,g,seed):
    ss=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(a,b,g))
    for tr,te in ss:
        assert not set(g[tr])&set(g[te])
        assert len(np.unique(b[tr]))==len(np.unique(b[te]))==2
    return ss
OUTER=[];splitrows=[]
for repeat in range(3):
    for fold,(tr,te) in enumerate(splits(X,y,groups,BASE_SEED+repeat)):
        OUTER.append((repeat,fold,tr,te))
        for side,ids in [("train",tr),("test",te)]:
            splitrows.extend(dict(repeat=repeat,fold=fold,side=side,row_id=int(D.row_id.iloc[i]),group_id=int(groups[i])) for i in ids)
pd.DataFrame(splitrows).to_csv(OUT/"02_внешние_разбиения.csv",index=False)
display(pd.DataFrame([dict(repeat=r,fold=f,train_n=len(tr),train_deaths=int(y[tr].sum()),test_n=len(te),test_deaths=int(y[te].sum())) for r,f,tr,te in OUTER]))


## Предобработка внутри обучения

Числовые признаки заполняются медианами, категории модами текущего обучения; LR масштабирует непрерывные признаки. Индикаторы, включая СРБ, вычисляются независимо от импутации по исходным значениям через MissingIndicator. Проверочная часть сохраняет исходный дисбаланс; отбор выживших и веса применяются внутри обучения. Проверяется наличие и правильность missing_crp во всех семействах и baseline при indicators=True, отсутствие всех флагов при indicators=False.

In [ ]:
def make_preprocessor(columns,family,indicators):
    nums=[c for c in columns if c in meta["continuous"]];cats=[c for c in columns if c not in nums]
    numeric_steps=[("impute",SimpleImputer(strategy="median",keep_empty_features=True))]
    if family=="LR":numeric_steps.append(("scale",StandardScaler()))
    categories=[[1,2,3,4] if c=="killip" else [0,1] for c in cats]
    transformers=[("numeric",Pipeline(numeric_steps),nums),("categorical",Pipeline([("impute",SimpleImputer(strategy="most_frequent",keep_empty_features=True)),("onehot",OneHotEncoder(categories=categories,handle_unknown="ignore",sparse_output=False))]),cats)]
    indicator_columns=list(columns)
    if indicators:transformers.append(("missing",MissingIndicator(features="all",error_on_new=False),indicator_columns))
    return ColumnTransformer(transformers,verbose_feature_names_out=True,sparse_threshold=0)
def sample_indices(a,b,strategy,seed):
    rng=np.random.default_rng(seed);dead=np.flatnonzero(b==1);alive=np.flatnonzero(b==0);target=min(4*len(dead),len(alive));median=float(a.age.median())
    allocations={}
    if strategy in ["all_real","class_weights"]: ids=np.arange(len(b))
    elif strategy=="random_4to1":ids=np.r_[dead,rng.choice(alive,target,replace=False)]
    else:
        age=np.where(a.age.isna(),"missing",np.where(a.age.to_numpy()>median,"older","younger"))
        killip=np.where(a.killip.isna(),"missing",np.where(a.killip.to_numpy()>=3,"3-4","1-2"))
        keys=np.array([str(k)+"|"+str(v) for k,v in zip(killip,age)])
        names,counts=np.unique(keys[alive],return_counts=True)
        quotas=target*counts/len(alive);take=np.floor(quotas).astype(int)
        order=np.argsort(-(quotas-take),kind="stable")
        for i in order[:target-int(take.sum())]:take[i]+=1
        assert take.sum()==target and np.all(take<=counts)
        chosen=[rng.choice(alive[keys[alive]==key],int(k),replace=False) for key,k in zip(names,take)]
        ids=np.r_[dead,*chosen];allocations={str(k):dict(available=int(n),selected=int(t)) for k,n,t in zip(names,counts,take)}
    assert set(dead).issubset(ids) and len(ids)==len(np.unique(ids))
    if "4to1" in strategy:assert sum(b[ids]==0)==target
    SAMPLING_AUDIT.append(dict(seed=seed,strategy=strategy,train_n=len(b),train_deaths=len(dead),selected_n=len(ids),selected_deaths=int(b[ids].sum()),age_median=median,strata=json.dumps(allocations)))
    return ids
def make_model(family,params,strategy,b,seed):
    weighted=strategy=="class_weights";ratio=float(sum(b==0)/sum(b==1))
    if family=="LR":return LogisticRegression(**params,l1_ratio=0,solver="lbfgs",max_iter=3000,class_weight="balanced" if weighted else None,random_state=seed)
    if family=="RF":return RandomForestClassifier(**params,class_weight="balanced" if weighted else None,random_state=seed,n_jobs=2)
    if family=="CatBoost":return CatBoostClassifier(**params,loss_function="Logloss",verbose=False,allow_writing_files=False,thread_count=2,random_seed=seed,auto_class_weights="Balanced" if weighted else None)
    return XGBClassifier(**params,objective="binary:logistic",eval_metric="logloss",tree_method="hist",n_jobs=2,random_state=seed,scale_pos_weight=ratio if weighted else 1.0)
def train(a,b,family,cfg,seed):
    pre=make_preprocessor(list(a.columns),family,cfg["indicators"])
    z=pre.fit_transform(a)
    assert np.isfinite(z).all()
    names=pre.get_feature_names_out().tolist()
    assert "numeric__crp" in names, names
    assert ("missing__missingindicator_crp" in names)==cfg["indicators"], names
    if cfg["indicators"]:assert np.array_equal(z[:,names.index("missing__missingindicator_crp")],a.crp.isna().astype(int).to_numpy())
    expected=["missing__missingindicator_"+c for c in a.columns] if cfg["indicators"] else []
    assert [name for name in names if name.startswith("missing__")]==expected
    ids=sample_indices(a,b,cfg["strategy"],seed)
    model=make_model(family,cfg["params"],cfg["strategy"],b,seed)
    model.fit(z[ids],b[ids]);assert list(model.classes_)==[0,1]
    return {"preprocessor":pre,"model":model,"features":list(a.columns),"family":family,"config":cfg,"run_id":RUN_ID}
def predict(artifact,a):
    p=artifact["model"].predict_proba(artifact["preprocessor"].transform(a))[:,1]
    assert np.isfinite(p).all() and np.all((p>=0)&(p<=1))
    return p
def threshold(b,p):
    fpr,tpr,t=roc_curve(b,p,drop_intermediate=False)
    score=(tpr+1-fpr)/2;valid=np.isfinite(t)
    mx=score[valid].max();ids=np.flatnonzero(valid&np.isclose(score,mx,atol=1e-12,rtol=0))
    idx=ids[np.argmax(tpr[ids])];return float(t[idx])
def metrics(b,p,t):
    h=p>=t;tn,fp,fn,tp=confusion_matrix(b,h,labels=[0,1]).ravel()
    return dict(AP=average_precision_score(b,p),ROC_AUC=roc_auc_score(b,p),Brier=brier_score_loss(b,p),sensitivity=tp/(tp+fn),specificity=tn/(tn+fp),precision=precision_score(b,h,zero_division=0),F1=f1_score(b,h,zero_division=0),accuracy=accuracy_score(b,h),TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),n=len(b),deaths=int(sum(b)),mean_probability=float(np.mean(p)))
def suggest(trial,family):
    if family=="LR":return {"C":trial.suggest_float("C",1e-3,100,log=True)}
    if family=="RF":return {"n_estimators":trial.suggest_int("n_estimators",200,800,step=100),"max_depth":trial.suggest_int("max_depth",2,8),"min_samples_leaf":trial.suggest_categorical("min_samples_leaf",[1,2,4,8,16]),"max_features":trial.suggest_categorical("max_features",["sqrt",0.5,1.0])}
    if family=="CatBoost":return {"iterations":trial.suggest_int("iterations",100,800,step=100),"depth":trial.suggest_int("depth",2,6),"learning_rate":trial.suggest_float("learning_rate",.01,.2,log=True),"l2_leaf_reg":trial.suggest_float("l2_leaf_reg",.01,100,log=True)}
    return {"n_estimators":trial.suggest_int("n_estimators",100,800,step=100),"max_depth":trial.suggest_int("max_depth",1,5),"learning_rate":trial.suggest_float("learning_rate",.01,.2,log=True),"min_child_weight":trial.suggest_float("min_child_weight",1,20),"subsample":trial.suggest_float("subsample",.6,1),"colsample_bytree":trial.suggest_float("colsample_bytree",.6,1),"reg_lambda":trial.suggest_float("reg_lambda",.001,100,log=True),"reg_alpha":trial.suggest_float("reg_alpha",0,5)}
def inner_predictions(a,b,g,family,cfg,ss,seed):
    p=np.full(len(b),np.nan);aps=[];bs=[]
    for k,(tr,va) in enumerate(ss):
        artifact=train(a.iloc[tr],b[tr],family,cfg,seed+k)
        p[va]=predict(artifact,a.iloc[va]);aps.append(average_precision_score(b[va],p[va]));bs.append(brier_score_loss(b[va],p[va]))
    assert np.isfinite(p).all()
    return p,float(np.mean(aps)),float(np.mean(bs))
def tune(a,b,g,family,ss,name,seed,trials=50,fixed=None):
    study_name=RUN_ID+"_"+name+"_"+family
    sampler=optuna.samplers.TPESampler(seed=seed,multivariate=False,constant_liar=False)
    study=optuna.create_study(study_name=study_name,storage=STORAGE,direction="maximize",sampler=sampler,load_if_exists=True)
    if not study.trials:
        options=[(s,i) for s in STRATEGIES for i in [False,True]] if fixed is None else [(fixed["strategy"],fixed["indicators"])]
        for s,i in options:study.enqueue_trial({**BASE[family],"strategy":s,"indicators":i})
        study.set_user_attr("config",CONFIG);study.set_user_attr("features",list(a.columns))
    def objective(trial):
        try:
            cfg={"strategy":trial.suggest_categorical("strategy",STRATEGIES) if fixed is None else fixed["strategy"],"indicators":trial.suggest_categorical("indicators",[False,True]) if fixed is None else fixed["indicators"],"params":suggest(trial,family)}
            p,ap,br=inner_predictions(a,b,g,family,cfg,ss,seed)
            trial.set_user_attr("config",cfg);trial.set_user_attr("Brier",br)
            return ap
        except Exception as exc:
            trial.set_user_attr("error",repr(exc));raise
    done=len([t for t in study.trials if t.state!=optuna.trial.TrialState.WAITING])
    if done<trials:study.optimize(objective,n_trials=trials-done,n_jobs=1,catch=(Exception,))
    complete=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE]
    assert complete,study_name
    maximum=max(t.value for t in complete);eligible=[t for t in complete if maximum-t.value<=1e-6]
    best=min(eligible,key=lambda t:(t.user_attrs["Brier"],t.number))
    cfg=best.user_attrs["config"]
    p,ap,br=inner_predictions(a,b,g,family,cfg,ss,seed)
    log(f"{name} {family}: {len(complete)}/{trials} complete; inner AP={ap:.5f}; {cfg}")
    return {"family":family,"config":cfg,"inner_AP":ap,"inner_Brier":br,"threshold":threshold(b,p),"study_name":study_name,"best_trial":best.number,"completed":len(complete)}
def choose(results):
    mx=max(r["inner_AP"] for r in results)
    return min([r for r in results if mx-r["inner_AP"]<=1e-6],key=lambda r:(r["inner_Brier"],FAMILIES.index(r["family"])))
def records(ids,r,f,family,p,t,cfg):
    return [dict(row_id=int(D.row_id.iloc[j]),group_id=int(groups[j]),repeat=r,fold=f,family=family,y_true=int(y[j]),p_death=float(prob),threshold=t,run_id=RUN_ID,configuration=json.dumps(cfg,sort_keys=True)) for j,prob in zip(ids,p)]


FEATURE_AUDIT=[]
for f in FAMILIES:
    for indicator_option in [False,True]:
        for drop in [[],["tag","ms"],["ms"],["tag"]]:
            columns=[c for c in FEATURES if c not in drop]
            processor=make_preprocessor(columns,f,indicator_option)
            z_check=processor.fit_transform(X.iloc[OUTER[0][2]][columns])
            names=processor.get_feature_names_out().tolist()
            assert "numeric__crp" in names and ("missing__missingindicator_crp" in names)==indicator_option
            assert len(names)==z_check.shape[1]
            assert sum(s.startswith("missing__") for s in names)==(len(columns) if indicator_option else 0)
            FEATURE_AUDIT.append(dict(run_id=RUN_ID,family=f,indicators=indicator_option,dropped=json.dumps(drop),n_features=len(names),crp_value_present=True,crp_missing_flag_present=indicator_option,transformed_features=json.dumps(names)))
pd.DataFrame(FEATURE_AUDIT).to_csv(OUT/"02_проверка_состава_признаков.csv",index=False)
print("CRP value retained; missing-CRP flag restored before imputation when indicators=True in every family and baseline")


## Вложенная оценка baseline, настроенных семейств и selector
Для каждого внешнего train создаётся общий inner split, заново выполняются четыре Optuna-поиска. Порог вычисляется по внутренним OOF выбранной конфигурации, максимум balanced accuracy; в равенстве выбирается большая чувствительность. Внешняя проверка сохраняет исходную частоту событий. Baseline — фиксированные параметры и без весов, с индикаторами; его порог также выбирается внутри train. Эффект настройки относится ко всему пайплайну.

In [ ]:
OOF=[];RESULTS=[];FOLD_METRICS=[]
for repeat,fold,tr,te in OUTER:
    seed=BASE_SEED+100*repeat+10*fold
    a=X.iloc[tr];b=y[tr];g=groups[tr];ss=splits(a,b,g,seed+1000)
    for k,(it,iv) in enumerate(ss):
        INNER_SPLITS.extend(dict(repeat=repeat,fold=fold,inner_fold=k,side=side,row_id=int(D.row_id.iloc[tr[j]]),group_id=int(g[j])) for side,ii in [("train",it),("validation",iv)] for j in ii)
    candidates=[]
    for family in FAMILIES:
        result=tune(a,b,g,family,ss,f"r{repeat}_f{fold}",seed)
        candidates.append(result);RESULTS.append(dict(repeat=repeat,fold=fold,**result))
        model=train(a,b,family,result["config"],seed)
        p=predict(model,X.iloc[te]);t=result["threshold"]
        OOF.extend(records(te,repeat,fold,family,p,t,result["config"]))
        FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family,**metrics(y[te],p,t)))
        cfg={"params":BASE[family],"strategy":"all_real","indicators":True}
        pi,_,_=inner_predictions(a,b,g,family,cfg,ss,seed);tb=threshold(b,pi)
        baseline=train(a,b,family,cfg,seed);pb=predict(baseline,X.iloc[te])
        OOF.extend(records(te,repeat,fold,family+"_baseline",pb,tb,cfg))
        FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family+"_baseline",**metrics(y[te],pb,tb)))
    winner=choose(candidates);family=winner["family"]
    model=train(a,b,family,winner["config"],seed);p=predict(model,X.iloc[te]);t=winner["threshold"]
    OOF.extend(records(te,repeat,fold,"selector",p,t,{"selected_family":family,**winner["config"]}))
    FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family="selector",selected_family=family,**metrics(y[te],p,t)))
    log(f"Outer r{repeat} f{fold} completed, selector={family}")
    pd.DataFrame(OOF).to_csv(OUT/"02_oof_predictions.csv",index=False)
    pd.DataFrame(FOLD_METRICS).to_csv(OUT/"02_метрики_блоков.csv",index=False)
    savejson(RESULTS,"02_результаты_поиска.json")
pd.DataFrame(INNER_SPLITS).to_csv(OUT/"02_внутренние_разбиения.csv",index=False)
pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/"02_аудит_балансировки.csv",index=False)
oof=pd.DataFrame(OOF);fm=pd.DataFrame(FOLD_METRICS)
assert oof.groupby(["repeat","family"]).row_id.nunique().eq(len(D)).all()
assert not oof.duplicated(["repeat","family","row_id"]).any()
display(fm.groupby("family")[["AP","ROC_AUC","Brier"]].agg(["mean","std","min","max"]))


## Финальный поиск на всей когорте
Семейство артефакта выбирается по внутреннему AP полного поиска, без выбора по внешним результатам. Метрики этого внутреннего поиска оптимистичны и не заменяют selector_oof. Финальная модель обучается на всей допустимой когорте.

In [ ]:
final_ss=splits(X,y,groups,BASE_SEED+9999)
FINAL_RESULTS=[tune(X,y,groups,f,final_ss,"final",BASE_SEED+9999) for f in FAMILIES]
FINAL=choose(FINAL_RESULTS);SELECTED_FAMILY=FINAL["family"]
artifact=train(X,y,SELECTED_FAMILY,FINAL["config"],BASE_SEED+9999)
artifact["threshold"]=FINAL["threshold"];artifact["source_data_sha256"]=meta["data_sha256"];artifact["versions"]=VERSIONS
joblib.dump(artifact,OUT/"02_final_model.joblib")
savejson({"run_id":RUN_ID,"selection":FINAL,"all_families":FINAL_RESULTS,"features":FEATURES,"interpretation":"full-cohort fitted artifact; independent quality belongs to nested selector"},"02_final_config.json")
print("FINAL",FINAL);display(pd.DataFrame(FINAL_RESULTS)[["family","inner_AP","inner_Brier","threshold","completed"]])


## Контролируемые условные эксперименты
Выбранное семейство фиксировано глобально; приведённые ниже оценки условны относительно этого выбора. Каждый вариант заново настраивается в train каждого внешнего блока, 20 trials, одинаковые пациенты и split/seed. Удаление ТАГ/МС включает их собственные индикаторы пропусков; производные ТАГ/МС отсутствуют. Диабет остаётся как компонент метаболического состояния. Результаты не изменяют финальное правило полного набора. Значения улучшения не интерпретируются причинно.

In [ ]:
BRANCHES={"all_real":{"strategy":"all_real","indicators":True,"drop":[]},"class_weights":{"strategy":"class_weights","indicators":True,"drop":[]},"stratified_4to1":{"strategy":"stratified_4to1","indicators":True,"drop":[]},"random_4to1":{"strategy":"random_4to1","indicators":True,"drop":[]},"no_missing_indicators":{"strategy":"all_real","indicators":False,"drop":[]},"without_tag_ms":{"strategy":"all_real","indicators":True,"drop":["tag","ms"]},"tag_only":{"strategy":"all_real","indicators":True,"drop":["ms"]},"ms_only":{"strategy":"all_real","indicators":True,"drop":["tag"]}}
EX_OOF=[];EX_METRICS=[];EX_CONFIG=[]
for repeat,fold,tr,te in OUTER:
    seed=BASE_SEED+100*repeat+10*fold;b=y[tr];g=groups[tr];ss=splits(X.iloc[tr],b,g,seed+1000)
    for branch,settings in BRANCHES.items():
        columns=[c for c in FEATURES if c not in settings["drop"]]
        a=X.iloc[tr][columns]
        result=tune(a,b,g,SELECTED_FAMILY,ss,f"conditional_{branch}_r{repeat}_f{fold}",seed,trials=CONFIG["conditional_trials"],fixed=settings)
        model=train(a,b,SELECTED_FAMILY,result["config"],seed);p=predict(model,X.iloc[te][columns]);t=result["threshold"]
        EX_OOF.extend(records(te,repeat,fold,branch,p,t,result["config"]))
        EX_METRICS.append(dict(repeat=repeat,fold=fold,branch=branch,**metrics(y[te],p,t)))
        EX_CONFIG.append(dict(repeat=repeat,fold=fold,branch=branch,**result))
    pd.DataFrame(EX_OOF).to_csv(OUT/"02_условные_oof.csv",index=False)
    pd.DataFrame(EX_METRICS).to_csv(OUT/"02_условные_метрики_блоков.csv",index=False)
    savejson(EX_CONFIG,"02_условные_конфигурации.json")
    log(f"Conditional outer r{repeat} f{fold} complete")
ex=pd.DataFrame(EX_OOF);em=pd.DataFrame(EX_METRICS)
display(em.groupby("branch")[["AP","ROC_AUC","Brier"]].agg(["mean","std"]))


## Сводки, калибровка и условная неопределённость
Основная сводка — среднее метрик по трём полным OOF-повторениям; дополнительно показаны 9 блоков (SD/диапазон не доверительный интервал). Bootstrap 1000 раз ресэмплирует группы пациентов, сохраняя все их записи и все повторения, а парные сравнения используют одинаковый отбор. Интервалы учитывают условную неопределённость фиксированных OOF-прогнозов, не весь процесс обучения и выбора. Нулевая модель предсказывает частоту смерти своего внешнего train и всех относит к выжившим.

In [ ]:
def summarize_oof(frame,key):
    rows=[]
    for (name,repeat),v in frame.groupby([key,"repeat"]):
        rows.append(dict(name=name,repeat=repeat,**metrics(v.y_true.to_numpy(),v.p_death.to_numpy(),v.threshold.to_numpy())))
    result=pd.DataFrame(rows)
    summary=result.groupby("name").agg(AP=("AP","mean"),AP_repeat_SD=("AP","std"),ROC_AUC=("ROC_AUC","mean"),Brier=("Brier","mean"),sensitivity=("sensitivity","mean"),specificity=("specificity","mean"),precision=("precision","mean"),F1=("F1","mean"),accuracy=("accuracy","mean"),mean_probability=("mean_probability","mean"))
    return result,summary
null=[]
for r,f,tr,te in OUTER:null.extend(records(te,r,f,"prevalence_baseline",np.full(len(te),y[tr].mean()),1.0,{"train_prevalence":float(y[tr].mean())}))
oof=pd.concat([oof,pd.DataFrame(null)],ignore_index=True)
oof.to_csv(OUT/"02_oof_predictions.csv",index=False)
repeats,summary=summarize_oof(oof,"family")
repeats.to_csv(OUT/"02_метрики_повторений.csv",index=False);summary.to_csv(OUT/"02_сравнение_моделей.csv")
exrepeats,exsummary=summarize_oof(ex,"family")
exrepeats.to_csv(OUT/"02_условные_метрики_повторений.csv",index=False);exsummary.to_csv(OUT/"02_условные_сравнения.csv")
# Fixed group resampling, same draws for every family/branch; no pseudoreplication.
unique=np.unique(groups);bygroup={g:np.flatnonzero(groups==g) for g in unique}
rng=np.random.default_rng(BASE_SEED+4444)
BOOT=[np.concatenate([bygroup[g] for g in rng.choice(unique,len(unique),replace=True)]) for _ in range(1000)]
def prediction_arrays(frame,names):
    result={}
    for name in names:
        v=frame[frame.family==name]
        result[name]=np.array([v[v.repeat==r].set_index("row_id").loc[D.row_id,"p_death"].to_numpy() for r in range(3)])
    return result
def boot_scores(arrays):
    scores={name:[] for name in arrays}
    for ids in BOOT:
        if len(np.unique(y[ids]))<2:continue
        for name,p in arrays.items():
            scores[name].append(np.mean([[average_precision_score(y[ids],pr[ids]),roc_auc_score(y[ids],pr[ids]),brier_score_loss(y[ids],pr[ids])] for pr in p],axis=0))
    return {k:np.array(v) for k,v in scores.items()}
bs=boot_scores(prediction_arrays(oof,FAMILIES+["selector"]))
ci=[]
for name,v in bs.items():
    for j,metric in enumerate(["AP","ROC_AUC","Brier"]):
        lo,hi=np.quantile(v[:,j],[.025,.975]);ci.append(dict(family=name,metric=metric,estimate=summary.loc[name,metric],lower=lo,upper=hi,scope="conditional group bootstrap; fixed OOF predictions"))
pd.DataFrame(ci).to_csv(OUT/"02_условные_bootstrap_CI.csv",index=False)
be=boot_scores(prediction_arrays(ex,list(BRANCHES)))
comparisons=[("class_weights","all_real"),("stratified_4to1","all_real"),("random_4to1","all_real"),("all_real","no_missing_indicators"),("all_real","without_tag_ms"),("tag_only","without_tag_ms"),("ms_only","without_tag_ms"),("all_real","tag_only"),("all_real","ms_only")]
deltas=[];fold_deltas=[]
for plus,minus in comparisons:
    for j,metric in enumerate(["AP","ROC_AUC","Brier"]):
        diff=be[plus][:,j]-be[minus][:,j];lo,hi=np.quantile(diff,[.025,.975])
        deltas.append(dict(plus=plus,minus=minus,metric=metric,delta=float(exsummary.loc[plus,metric]-exsummary.loc[minus,metric]),lower=float(lo),upper=float(hi),scope="conditional on globally chosen family and fixed paired OOF"))
    a=em[em.branch==plus].set_index(["repeat","fold"]);b=em[em.branch==minus].set_index(["repeat","fold"])
    for (r,f),v in (a[["AP","ROC_AUC","Brier"]]-b[["AP","ROC_AUC","Brier"]]).iterrows():fold_deltas.append(dict(plus=plus,minus=minus,repeat=r,fold=f,**v.to_dict()))
delta=pd.DataFrame(deltas);delta.to_csv(OUT/"02_парные_изменения.csv",index=False);pd.DataFrame(fold_deltas).to_csv(OUT/"02_парные_изменения_блоков.csv",index=False)
cal=[]
for (family,r),v in pd.concat([oof.assign(kind="main"),ex.assign(kind="conditional")],ignore_index=True).groupby(["family","repeat"]):
    v=v.copy();v["bin"]=pd.qcut(v.p_death,5,duplicates="drop")
    for bn,g in v.groupby("bin",observed=True):cal.append(dict(family=family,repeat=r,bin=str(bn),n=len(g),deaths=int(g.y_true.sum()),mean_prediction=g.p_death.mean(),event_rate=g.y_true.mean()))
cal=pd.DataFrame(cal);cal.to_csv(OUT/"02_калибровка.csv",index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for family in FAMILIES+["selector"]:
    v=cal[cal.family==family]
    axes[0].plot(v.mean_prediction,v.event_rate,"o",alpha=.65,label=family)
    axes[1].bar(family,summary.loc[family,"AP"],color="#35658c")
axes[0].plot([0,1],[0,1],"--",color="grey");axes[0].set(xlabel="Средний прогноз смерти",ylabel="Частота смерти",title="Калибровка: 5 квантильных групп × 3 повторения");axes[0].legend()
axes[1].axhline(y.mean(),ls="--",color="grey",label="Частота смерти");axes[1].set(ylabel="AP смерти",title="Средний AP полных внешних OOF-повторений");axes[1].legend();figsave("02_качество_и_калибровка")
for family in FAMILIES:
    plt.figure(figsize=(6,5));v=cal[cal.family==family]
    for r,g in v.groupby("repeat"):plt.plot(g.mean_prediction,g.event_rate,"o-",label=f"Повторение {r+1}")
    plt.plot([0,1],[0,1],"--",color="grey");plt.xlabel("Прогноз смерти");plt.ylabel("Частота смерти");plt.title(family+": калибровка на реальных внешних пациентах");plt.legend();figsave("02_калибровка_"+family)
plt.figure(figsize=(10,5));em.groupby("branch").AP.mean().plot.bar(color="#35658c");plt.ylabel("AP: среднее внешних блоков");plt.title("Условное сравнение обработки и ТАГ/МС");figsave("02_условные_эксперименты")
trials_frames=[]
for study in optuna.get_all_study_summaries(storage=STORAGE):
    if study.study_name.startswith(RUN_ID):
        s=optuna.load_study(study_name=study.study_name,storage=STORAGE);t=s.trials_dataframe();t.insert(0,"study",study.study_name);trials_frames.append(t)
trials_df=pd.concat(trials_frames,ignore_index=True);trials_df.to_csv(OUT/"02_trials.csv",index=False)
pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/"02_аудит_балансировки.csv",index=False)
savejson({"run_id":RUN_ID,"protocol":CONFIG,"data_sha256":sha(OUT/"01_данные.csv"),"model_sha256":sha(OUT/"02_final_model.joblib"),"studies":len(trials_frames),"trial_states":trials_df.state.value_counts().to_dict(),"completed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()},"02_происхождение_результатов.json")
log("All nested, final, controlled experiments completed")
display(summary);display(exsummary);display(delta)


## Отчёт по фактическим расчётам
Отчёт ниже генерируется из сохранённых результатов текущего RUN_ID. Качество выбранного артефакта не приписывается оптимистично одному семейству: независимая оценка алгоритма выбора — selector.

In [ ]:
sel=summary.loc["selector"]
text=f"""# Четыре модели, Optuna и дополнительные эксперименты ОКС

Бинарный индикатор отсутствия СРБ возвращён во все модели и baseline с indicators=True. Он вычисляется по исходной неполноте до заполнения пропусков, как остальные индикаторы. Числовой СРБ и обучаемая импутация сохранены. В ветке без индикаторов отсутствуют все флаги. Проверка выполняется при каждом обучении.

Статус: выполнено. RUN_ID `{RUN_ID}`. Использована среда analytics_learning, Python (Analytics); все расчёты новые, исходные outputs не применялись.

## Данные и независимый протокол

906 госпитализаций, 43 смерти; {D.group_id.nunique()} консервативных групп пациентов. 3 внешних групповых стратифицированных блока × 3 повторения, в каждом 3 внутренних блока. Разбиения общие для четырёх семейств; повторный пациент не пересекает train/test. Все преобразования обучались только в train, проверочная часть не балансировалась. Для каждого семейства выполнялись 50 Optuna-trials в каждом из 9 внешних обучений и в финальном поиске. TPESampler(seed), последовательные trials; у моделей максимум 2 потока, early stopping отсутствует. Все состояния сохранены в SQLite и CSV. Сравнение настройки относится к всему пайплайну, включая обработку, и не изолирует эффект одних параметров модели.

Средний внутренний AP положительного класса смерти — критерий выбора. При равенстве в пределах 1e-6: меньший Brier, затем LR→RF→CatBoost→XGBoost. Порог максимизирует balanced accuracy внутренних OOF; при равенстве выше чувствительность. Четыре baseline заранее зафиксированы. Качество полной процедуры выбора оценено отдельными selector_oof; внешние метрики не используются для выбора финального семейства.

## Основные численные результаты

Среднее метрик по 3 полным OOF-повторениям:

{summary.reset_index().to_markdown(index=False,floatfmt='.5f')}

[Метрики всех 9 блоков](02_метрики_блоков.csv), [метрики повторений](02_метрики_повторений.csv), [OOF](02_oof_predictions.csv), [условные интервалы](02_условные_bootstrap_CI.csv). SD и диапазон блоков не являются 95% CI. Bootstrap 1000 раз ресэмплирует группы пациентов совместно во всех повторениях и учитывает только неопределённость фиксированных OOF-прогнозов, без повторного обучения/выбора. Матрицы ошибок и чувствительность/специфичность/precision/F1 доступны для каждого повторения и блока.

Независимая оценка полной процедуры: AP={sel.AP:.5f}, ROC-AUC={sel.ROC_AUC:.5f}, Brier={sel.Brier:.5f}. Финальный артефакт: {SELECTED_FAMILY}, выбранный по внутреннему поиску на всей когорте. Его обучающие или внутренние метрики не считаются независимыми.

"""
for f in FAMILIES:
    fin=next(z for z in FINAL_RESULTS if z["family"]==f);v=summary.loc[f]
    text+=f"""## {f}

Пространство поиска: {CONFIG['spaces'][f]}. Baseline: `{json.dumps(BASE[f])}`. В каждом основном study 50 попыток; окончательный study `{fin['study_name']}`, завершено {fin['completed']}. Лучшие параметры поиска на всей когорте: `{json.dumps(fin['config'],ensure_ascii=False)}`. Внутренний AP={fin['inner_AP']:.5f}, Brier={fin['inner_Brier']:.5f}; это значения после настройки. Внешние средние AP={v.AP:.5f}, ROC-AUC={v.ROC_AUC:.5f}, Brier={v.Brier:.5f}; чувствительность={v.sensitivity:.5f}, специфичность={v.specificity:.5f}, precision={v.precision:.5f}. Калибровка: [рисунок](02_калибровка_{f}.png), [таблица наблюдений/смертей](02_калибровка.csv). Общие файлы параметров/trials: [поиск](02_результаты_поиска.json), [trials](02_trials.csv), [SQLite](02_optuna_missing_crp_возвращён.sqlite). При 43 событиях результат имеет большую неопределённость; лучший внутренний trial сам по себе не устанавливает превосходство над другими семействами.

"""
text+=f"""## Дисбаланс и пропуски

Предобработка: непрерывные признаки — медианы; категории — мода train, one-hot фиксированных клинических кодов, неизвестная категория допустима; полностью пустой столбец — 0 с сохранением схемы. LR стандартизует непрерывные признаки. Индикаторы — отдельные бинарные флаги отсутствия значений каждого исходного признака, включая СРБ, вычисленные до импутации. Веса определяются по текущему обучению. Стратифицированный undersampling включает Killip 1–2/3–4/неизвестен и возраст выше/не выше медианы текущего train; пропуск возраста — отдельная страта. Квоты пропорциональны структуре выживших, метод наибольших остатков обеспечивает ровно 4 выживших на смерть при наличии достаточного большинства. Все обучающие смерти сохранены, отбор без возвращения. Параметры преобразований оценивались на полном реальном train до сокращения. Взвешивание и отбор одновременно не применялись. Дополнительная посткалибровка не выполнялась, поэтому выходы взвешенных/сокращённых моделей не объявляются готовыми клиническими рисками.

Условные эксперименты фиксируют глобально выбранное семейство {SELECTED_FAMILY}; по 20 trials на каждую из 8 ветвей и каждое внешнее обучение. Меньший отдельный бюджет дополнительных экспериментов был зафиксирован до запуска для вычислительной выполнимости. Он одинаков между сравниваемыми ветвями; это не оценка полной процедуры выбора. Основной полный протокол и финальный набор признаков по этим сравнениям не изменяются.

{exsummary.reset_index().to_markdown(index=False,floatfmt='.5f')}

Сравнение all_real / class_weights / stratified_4to1 / random_4to1 относится к индикаторам, включённым во всех этих ветвях. Сравнение all_real и no_missing_indicators фиксирует отсутствие весов и одну схему импутации, обе версии заново настраиваются. Сохранён [аудит каждого обучающего отбора](02_аудит_балансировки.csv), включающий численности, медиану возраста, доступность и квоты страт. [Конфигурации условных studies](02_условные_конфигурации.json), [OOF](02_условные_oof.csv), [таблица](02_условные_сравнения.csv).

## Добавочная прогнозная информация ТАГ и МС

Клинический набор одинаков; удаляются непосредственно ТАГ/МС и их индикаторы. Производных ТАГ/МС в наборе нет. Диабет остаётся, а правило постановки МС отсутствует: информация о компонентах МС полностью не устраняется. Проверены полный набор, без обоих, только ТАГ и только МС; каждую версию заново настроили с одинаковым бюджетом и одинаковыми группами/разбиениями. Изменения ниже: плюс минус; улучшение AP/AUC соответствует положительному знаку, улучшение Brier отрицательному. Интервалы условны, парность и группы сохранены.

{delta.to_markdown(index=False,floatfmt='.5f')}

[Парные изменения](02_парные_изменения.csv), [изменения каждого блока](02_парные_изменения_блоков.csv). Эти данные описывают добавочную прогнозную информацию конкретного согласованного набора; они не устанавливают причинный эффект и не доказывают отсутствие клинической связи при нулевом/отрицательном изменении AP.

## Финальный артефакт и ограничения

`{json.dumps(FINAL,ensure_ascii=False)}`. [Модель вместе с преобразованиями](02_final_model.joblib), [точная конфигурация](02_final_config.json), [протокол](02_протокол.json), [происхождение](02_происхождение_результатов.json). Фиксированный полный набор включает 15 характеристик; обучающая процедура применена к полной когорте 906 записей; число записей классификатора определяется выбранной стратегией дисбаланса. Независимое качество относится к selector, который может выбирать разные семейства в разных блоках. Условные внешние сравнения окончательного семейства не устраняют зависимость его глобального выбора от данных.

Основные ограничения: 43 события, одна ретроспективная когорта, отсутствие внешней/временной валидации, неизвестное время измерения максимального ТАГ и лечения, качество регистрации отрицательных диагнозов и группировки по ФИО. Нельзя переносить этот результат на прогноз при поступлении или интерпретировать SHAP причинно. Проверка калибровки проведена на естественной частоте смерти; существенные отклонения следует учитывать перед клиническим применением. Сверхпараметры и ветви поиска не гарантируют нахождения глобального оптимума.

## Материал для статьи

Четыре семейства классификаторов сопоставлялись в повторной вложенной групповой стратифицированной кросс-валидации с 3 внешними и 3 внутренними блоками. В каждом внешнем обучении гиперпараметры и обработка неполноты/дисбаланса подбирались заново методом TPE в Optuna по среднему Average Precision положительного класса госпитальной смерти. Внешние пациенты оставались реальными и сохраняли естественную частоту события. Отдельно оценивался алгоритм выбора семейства по внутренним данным: средние AP, ROC-AUC и Brier составили {sel.AP:.4f}, {sel.ROC_AUC:.4f} и {sel.Brier:.4f}. Для окончательного артефакта повторный внутренний поиск на полной когорте выбрал {SELECTED_FAMILY}. Дополнительные парные эксперименты на одинаковых внешних наблюдениях оценивали стратегии работы с дисбалансом, индикаторы пропусков и добавление ТАГ/МС; их результаты являются условными относительно выбранного семейства и не подменяют независимую оценку полной процедуры.

Воспроизведение: выполнить 01→02→03 с чистыми ядрами Python (Analytics). SQLite содержит завершённые trials текущего RUN_ID; при неизменных отпечатках/коде studies могут быть повторно загружены, но внутренние и внешние прогнозы пересчитываются. Смена кода/конфигурации создаёт другой RUN_ID. Внутренние/внешние разбиения, seeds, параметры и версии сохранены рядом. Источники не изменялись.
"""
(OUT/"02_модели_Optuna_и_эксперименты.md").write_text(text)
print("Report written from current computations.")
